In [ ]:
from config import settings
from langchain_experimental.graph_transformers import LLMGraphTransformer
from langchain_community.graphs import Neo4jGraph
from langchain_openai import ChatOpenAI

In [ ]:
llm = ChatOpenAI(
    model="gpt-4o",
    api_key=settings.OPENAI_API_KEY,
)

llm_transformer = LLMGraphTransformer(llm=llm)

In [ ]:
from IPython.display import Markdown, display

def display_markdown(text):
    """
    Display the given text in markdown format in a Jupyter notebook.

    Parameters:
    text (str): The text to be displayed in markdown format.
    """
    display(Markdown(text))

In [ ]:
graph = Neo4jGraph(
    url=settings.NEO4J_URI,
    password=settings.NEO4J_PASSWORD,
    username=settings.NEO4J_USERNAME
)

In [ ]:
from yfiles_jupyter_graphs import GraphWidget
from neo4j import GraphDatabase

default_cypher = "81`    "

def showGraph(cypher: str = default_cypher):
    driver = GraphDatabase.driver(
        uri = settings.NEO4J_URI,
        auth = (settings.NEO4J_USERNAME,
                settings.NEO4J_PASSWORD)
    )
    session = driver.session()
    widget = GraphWidget(graph = session.run(cypher).graph())
    widget.node_label_mapping = 'id'
    #display(widget)
    return widget

In [ ]:
# Load data into the graph database using the dataloader
from data_loader import load_data_into_database

load_data_into_database()

showGraph()

In [ ]:
from langchain_community.vectorstores import Neo4jVector
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key=settings.OPENAI_API_KEY,
)

vector_index = Neo4jVector.from_existing_graph(
    embeddings,
    search_type="hybrid",
    node_label="Document",
    text_node_properties=["text"],
    embedding_node_property="embedding",
    index_name = "document_embedding",
)

In [ ]:
graph.query("""
  SHOW VECTOR INDEXES
  """
)

##### Using GraphCypherQA


In [ ]:
graph.refresh_schema()
graph.get_structured_schema

In [ ]:
from langchain_community.chains.graph_qa.cypher import GraphCypherQAChain

cypher_chain = GraphCypherQAChain.from_llm(graph=graph, llm=llm, verbose=True)
response = cypher_chain.invoke({"query": "How many submissions are there in total? Please group this submission into distinct talks and tutorials?"})
response

In [ ]:
response = cypher_chain.invoke({"query": "What are the various locations where the conference is holding?"})
response

In [ ]:
from langchain.chains.retrieval_qa.base import RetrievalQA

qa_graph_chain = RetrievalQA.from_chain_type(
    llm, retriever=vector_index.as_retriever(), verbose = True
)

result = qa_graph_chain.invoke({"query": "Where and when is the talk about Graph database holding? Who is the speaker?"})
result["result"]

In [ ]:
from langchain.agents import AgentExecutor, create_react_agent
from langchain.tools import Tool
from langchain import hub
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import Neo4jChatMessageHistory

tools = [
    Tool.from_function(
        name="Cypher Chain",
        description="For when you need to answer questions that can be answered by directly querying the database without having to use knowledge of vector embeddings. The question will be a string. Return a string.",
        func=cypher_chain.invoke,
    ),

    Tool.from_function(
        name="Natural QA Chain",
        description="For answering text based questions that are not easy or straight-forward to answer using Cypher queries but can be answered using vector embeddings. The question will be a String. Return a String",
        func=qa_graph_chain.invoke
    )
]


def get_memory(session_id):
    return Neo4jChatMessageHistory(session_id=session_id, graph=graph)


agent_prompt = hub.pull("hwchase17/react-chat")
agent = create_react_agent(llm, tools, agent_prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, handle_parsing_errors=True)

chat_agent = RunnableWithMessageHistory(
    agent_executor,
    get_memory,
    input_messages_key="input",
    history_messages_key="chat_history",
)

In [ ]:
question = "I am a software engineer and I would like to learn more about graph databases and retrieval augmented generation, which talk should I attend?"
response = chat_agent.invoke(
    {
        "input": question
    },
    {"configurable": {"session_id": "random_id"}},
)
response['output']

#### Load more data

In [ ]:
from langchain_core.documents import Document
from data_loader import scrape_website, ScrapedWebsite, generate_md5_hash
from langchain.text_splitter import TokenTextSplitter
from tqdm import tqdm

list_of_websites = ["https://pydata.org/london2024", "https://pydata.org/london2024/about",
                    "https://pydata.org/london2024/job-board", "https://pydata.org/london2024/humble", "https://pydata.org/london2024/sponsor"]

scraped_data: list[ScrapedWebsite] = [scrape_website(url) for url in tqdm(list_of_websites, desc="Scraping website")]
list_of_documents: list[Document] = [
    Document(
        page_content=data.content,
        metadata={"title": data.title,
                  "url": data.url,
                  "id": generate_md5_hash(data.content)
                  })
    for data in tqdm(scraped_data, desc="Processing documents")]

text_splitter = TokenTextSplitter(chunk_size=1024, chunk_overlap=24)
documents = text_splitter.split_documents(list_of_documents)

print(documents)

Please uncomment the code below to run the LLMGraphTransformer. This transforms raw text into graph document.

In [ ]:
# graph_documents = llm_transformer.convert_to_graph_documents(documents)
# 
# print(f"Graph Documents: {graph_documents}")

In [ ]:
# import pickle
# 
# with open("data/graph_documents.pkl", "wb") as f:
#     pickle.dump(graph_documents, f)

In [ ]:
import pickle
with open("data/graph_documents.pkl", "rb") as f:
    graph_documents = pickle.load(f)

In [ ]:
graph_documents

In [ ]:
from modified_neo4j_graph import ModifiedNeo4JGraph
graph = ModifiedNeo4JGraph(
    url=settings.NEO4J_URI,
    password=settings.NEO4J_PASSWORD,
    username=settings.NEO4J_USERNAME
)
graph.add_graph_documents_with_embeddings(
    graph_documents=graph_documents,
    baseEntityLabel=True,
    include_source=True,
    embedding=embeddings
)

In [ ]:
from typing import Callable

def get_chat_agent(cypher_chain: Callable, qa_graph_chain: Callable) -> Callable:
    tools = [
        Tool.from_function(
            name="Cypher Chain",
            description="For when you need to answer questions that can be answered by directly querying the database without having to use knowledge of vector embeddings. The question will be a string. Return a string.",
            func=cypher_chain.invoke,
        ),
    
        Tool.from_function(
            name="Natural QA Chain",
            description="For answering text based questions that are not easy or straight-forward to answer using Cypher queries but can be answered using vector embeddings. The question will be a String. Return a String",
            func=qa_graph_chain.invoke
        )
    ]
    
    agent_prompt = hub.pull("hwchase17/react-chat")
    agent = create_react_agent(llm, tools, agent_prompt)
    agent_executor = AgentExecutor(agent=agent, tools=tools, handle_parsing_errors=True)
    
    chat_agent = RunnableWithMessageHistory(
        agent_executor,
        get_memory,
        input_messages_key="input",
        history_messages_key="chat_history",
    )
    
    return chat_agent
    

def get_memory(session_id):
    return Neo4jChatMessageHistory(session_id=session_id, graph=graph)


In [ ]:
vector_index_without_retrieval_query = Neo4jVector.from_existing_graph(
    embeddings,
    search_type="hybrid",
    node_label="Document",
    text_node_properties=["text"],
    embedding_node_property="embedding",
    index_name = "document_embedding",
)

In [ ]:
qa_graph_chain_without_retrieval_query = RetrievalQA.from_chain_type(
    llm, retriever=vector_index_without_retrieval_query.as_retriever(), verbose = True
)

result = qa_graph_chain.invoke({"query": "Who are the keynote speakers?"})
display_markdown(result["result"])

In [ ]:
chat_agent = get_chat_agent(cypher_chain,qa_graph_chain_without_retrieval_query)
question = "What are the keynote speakers talking about?"
response = chat_agent.invoke(
    {
        "input": question
    },
    {"configurable": {"session_id": "randomId"}},
)
display_markdown(result['result'])

In [ ]:
vector_index_with_retrieval_query = Neo4jVector.from_existing_graph(
    embeddings,
    search_type="hybrid",
    node_label="Document",
    text_node_properties=["text"],
    embedding_node_property="embedding",
    index_name = "document_embedding",
    retrieval_query="""
    WITH node AS doc, score as similarity
    CALL {
        WITH doc
        MATCH (doc)-[*1..2]-(related)
        WITH doc, related, [key in keys(related) WHERE key <> 'embedding'] AS filtered_keys
        RETURN DISTINCT related, [key IN filtered_keys | key + ': ' + toString(related[key])] AS properties_list
    }
    WITH doc, similarity, properties_list
    RETURN coalesce(doc.text, '') + ' ' + coalesce(reduce(s = '', prop IN properties_list | s + ', ' + prop), '') AS text, similarity AS score, {source: doc.text} AS metadata
    LIMIT 25
    """
)

In [ ]:
qa_graph_chain = RetrievalQA.from_chain_type(
    llm, retriever=vector_index_with_retrieval_query.as_retriever(), verbose = True
)

result = qa_graph_chain.invoke({"query": "Which speakers are software engineers? "})
display_markdown(result["result"])

The above error is expected because in the retrieval query we are fetching more than enough context that we are passing to the LLM. Try reducing the limit in the retrieval query

In [ ]:
# graph.refresh_schema()
# graph.get_structured_schema

In [ ]:
SESSION_ID = "6bdabff3-2bb2-4f23-a696-e10435dc111de"
print(f"Session ID: {SESSION_ID}")

# while True:
q = input("> ")
print(f"Question: {q}")

response = chat_agent.invoke(
    {
        "input": q
    },
    {"configurable": {"session_id": SESSION_ID}},
)

print(f"Result: {response['output']}")

### Please Subscribe to my Newsletter

In [ ]:
from IPython.display import Image, display

display(Image(filename='images/BeatByteNewsletter.png', width=400))